# 05 - Training

Training of the convolutional-recurrent classifier (`wqu.models.cred`) on one representation, with the parameters of
the `training` and `model` sections of `configs/default.yaml`:
- input standardized per frequency with mean and standard deviation of the training records;
- weighted BCE loss, bad-class weight `n_good / n_bad` of the training records;
- Adam, learning rate reduced on plateau of the validation loss, early stopping, best model kept;
- per-epoch history (loss, AUC, balanced accuracy, ECE, learning rate) and its figures;
- MC dropout predictions on the validation set.

Every run writes to its own folder `training.runs_dir/<representation>_<date>_<time>/`: copy of the configuration,
normalization statistics, `history.csv` (updated at every epoch), `best_model.pt`, figures and validation predictions.

The training metrics are computed on the outputs produced while training (dropout active, weights changing within the
epoch), the validation metrics on a deterministic pass at the end of the epoch: with strong dropout the training loss can
stay above the validation loss without any anomaly.

## 0. Setup

In [ ]:
from pathlib import Path

import pandas as pd

from wqu.config import load_config
from wqu.data.loading import CV_COMPONENTS
from wqu.data.splitting import SPLIT_COLUMN, TRAIN, VALIDATION, attach_split, load_split
from wqu.models.cred import build_model
from wqu.models.loss import bad_class_weight, weighted_bce_loss
from wqu.models.uncertainty import prediction_table
from wqu.preprocessing.pipeline import FEATURE_INDEX_COLUMN, feature_file_path, read_feature_records
from wqu.training.data import FeatureDataset, frequency_statistics, make_loader
from wqu.training.metrics import classification_metrics
from wqu.training.trainer import VALIDATION_PREDICTIONS_FILE, create_run_dir, fit, select_device, set_seed
from wqu.visualization import apply_plot_style, plot_training_curve, plot_training_history, save_figure
import matplotlib.pyplot as plt

CONFIG_PATH = Path("..") / "configs" / "default.yaml"

config = load_config(CONFIG_PATH)
columns = config.columns
training = config.training
apply_plot_style()
set_seed(training.random_seed)
device = select_device(training.device)

features_path = feature_file_path(config.preprocessing, training.feature_source)
records = read_feature_records(features_path, columns.text)
records[SPLIT_COLUMN] = attach_split(records, load_split(config.split.output_file, columns), columns)
print(f"Feature file: {features_path} ({len(records):,} records); representation: {training.representation}; "
      f"device: {device}")

## 1. Data

Per-frequency statistics from the training records only, bad-class weight from the training labels.

In [ ]:
train_records = records[records[SPLIT_COLUMN] == TRAIN]
validation_records = records[records[SPLIT_COLUMN] == VALIDATION]
statistics = frequency_statistics(features_path, training.representation, train_records[FEATURE_INDEX_COLUMN].to_numpy())
bad_weight = bad_class_weight(train_records[columns.label].to_numpy())

run_dir = create_run_dir(training.runs_dir, training.representation, CONFIG_PATH)
statistics.save(run_dir / "normalization.npz")


def loader_for(subset: pd.DataFrame, shuffle: bool):
    """Loader of standardized images for a subset of the records."""
    dataset = FeatureDataset(features_path, training.representation, subset[FEATURE_INDEX_COLUMN].to_numpy(),
                             subset[columns.label].to_numpy(), statistics)
    return make_loader(dataset, training.batch_size, shuffle, training.num_workers, training.random_seed)


train_loader = loader_for(train_records, shuffle=True)
validation_loader = loader_for(validation_records, shuffle=False)
summary = records.groupby(SPLIT_COLUMN)[columns.label].agg(records="size", bad_rate="mean")
print(f"Run folder: {run_dir}")
print(f"Bad-class weight: {bad_weight:.3f}")
summary.round(3)

## 2. Training

The progress is printed at every epoch; `history.csv` in the run folder is rewritten at every epoch, so the run can be
followed from outside the notebook.

In [ ]:
model = build_model(config.model, len(CV_COMPONENTS), len(statistics.mean))
result = fit(model, train_loader, validation_loader, weighted_bce_loss(bad_weight), training, bad_weight, run_dir, device)
history = result.history
print(f"Best epoch: {result.best_epoch} of {len(history)}; stopped early: {result.stopped_early}; "
      f"total time {history['seconds'].sum() / 60:.1f} min")
history.round(4)

## 3. Training curves

Overview (loss and learning rate on top; AUC, balanced accuracy and ECE below) and the loss alone. The dashed line marks
the epoch of the saved model.

In [ ]:
figures_dir = run_dir / "figures"
overview = plot_training_history(history, result.best_epoch, f"Training history: {run_dir.name}")
save_figure(overview, figures_dir / "training_history.png")
save_figure(overview, figures_dir / "training_history.pdf")

loss_figure, loss_ax = plt.subplots(figsize=(7, 4.5), constrained_layout=True)
plot_training_curve(history, "loss", result.best_epoch, loss_ax)
save_figure(loss_figure, figures_dir / "training_loss.png")
save_figure(loss_figure, figures_dir / "training_loss.pdf")

## 4. Validation predictions with MC dropout

For every validation record: probability of the bad class from the deterministic pass (dropout off) and its entropy;
from `model.mc_samples` stochastic passes, mean and standard deviation of the probability, predictive entropy and its
split into aleatoric part and mutual information (epistemic part); both probabilities also corrected to the class
frequencies of the training set. The table is the input of notebook 06 (evaluation).

In [ ]:
predictions = prediction_table(model, validation_loader, config.model.mc_samples, device, bad_weight)
predictions.insert(0, "record", validation_records.index.to_numpy())
predictions.insert(1, columns.label, validation_records[columns.label].to_numpy())
predictions.to_csv(run_dir / VALIDATION_PREDICTIONS_FILE, index=False)

labels = predictions[columns.label].to_numpy()
comparison = {}
for name, probability in (("deterministic", "probability_deterministic"), ("mc_dropout", "probability_mean")):
    metrics = classification_metrics(predictions[probability].to_numpy(), labels, training.calibration_bins)
    metrics["ece"] = classification_metrics(predictions[f"{probability}_corrected"].to_numpy(), labels,
                                            training.calibration_bins)["ece"]
    comparison[name] = metrics
comparison = pd.DataFrame(comparison)
comparison.to_csv(run_dir / "validation_metrics.csv")
comparison.round(4)

In [ ]:
predictions.groupby(columns.label)[["probability_mean", "probability_std", "predictive_entropy",
                                     "mutual_information"]].describe().T.round(4)